# Improving Context

> Scotland’s nature is ...
> 
> Nature gives us ...
> 
> ... important nature and landscape interests ...
>
> ... connect with nature and create moments ...


```mermaid
flowchart LR


sc(scotlands) --> na(nature)
na -- "scotlands, nature"--> is(is)

st("[START]") --> na(nature)
na -- "[START], nature" --> gi(gives)

im(important) --> na(nature)
na -- "important, nature"--> an(and)
an -- "nature, and" --> la(landscape)

wi(with) --> na
na -- "with, nature"--> an(and)
an -- "nature, and" --> cr(creates)
```

In [1]:
import pickle
from itertools import accumulate
from random import randint
import ipywidgets as widgets
from IPython.display import display, Markdown


def upsert(into_model, from_token, to_token):

    # Not seen our "from" before? Create the entry.
    if from_token not in into_model:
        into_model[from_token] = {to_token: 1}
        return

    # Seen the "from", but not the "to"? Add this as an option.
    if to_token not in into_model[from_token]:
        into_model[from_token][to_token] = 1
    # Seen the "from" and the "to"? Add weight to this option.
    else:
        into_model[from_token][to_token] += 1

In [2]:
sentences_words = pickle.load(open('sentences_words.pkl', 'rb'))

two_model = {}
# Loop over every sentence in every blog.
for sentence in sentences_words:
    tokens = (
        [
            'START',
            'START',
        ]
        + sentence
        + ['END', 'END']
    )
    # For each word "from, from" & "to" pair, add them to the model.
    for idx in range(0, len(tokens) - 2):
        token_prev = ' '.join([tokens[idx], tokens[idx + 1]])
        token_this = tokens[idx + 2]
        upsert(two_model, token_prev, token_this)

# Save the model for re-use.
pickle.dump(two_model, open('models/forward_2.pkl', 'wb'))

In [3]:
def next_word(in_model, this_word):
    # Find all the "to" words for our "from".
    candidates = in_model[this_word].items()
    words = [v[0] for v in candidates]
    scores = [v[1] for v in candidates]

    # Score each "to" word based on it's weight, and line it up.
    # e.g 1 = A, 2-6 = B, 7-9 = C, etc.
    upper_lim = sum(scores)
    accumulated_scores = accumulate(scores)
    choices = zip(accumulated_scores, words, strict=True)

    # Roll a dice based on the maximum number.
    choice = randint(0, upper_lim)

    # Pick the one that matches our dice roll.
    chosen = list(filter(lambda c: c[0] >= choice, choices))[0][1]
    return chosen


# Pick a likely "first" word for a sentence.
word = widgets.Output(layout={'padding': '16px'})
with word:
    display(Markdown('## ' + next_word(two_model, 'START START')))

word

Output(layout=Layout(padding='16px'))

In [4]:
def generate_two_word_context_sentence(in_model):
    last_words = ['START', 'START']
    sentence = []
    for _ in range(55):
        word = next_word(in_model, ' '.join(last_words))
        if word == 'END':
            break
        sentence.append(word)
        last_words = [last_words[-1], word]

    return sentence


# Generate a random sentence.
generated_sentence = ' '.join(generate_two_word_context_sentence(two_model)) + '.'
two_sentence = widgets.Output(layout={'padding': '16px'})
with two_sentence:
    display(Markdown('## ' + generated_sentence.capitalize()))

two_sentence

Output(layout=Layout(padding='16px'))

# Where two words are good, are three better?

In [5]:
three_model = {}
# Loop over every sentence in every blog.
for sentence in sentences_words:
    tokens = (
        [
            'START',
            'START',
            'START',
        ]
        + sentence
        + ['END', 'END', 'END']
    )
    # For each word "from, from" & "to" pair, add them to the model.
    for idx in range(0, len(tokens) - 3):
        token_prev = ' '.join([tokens[idx], tokens[idx + 1], tokens[idx + 2]])
        token_this = tokens[idx + 3]
        upsert(three_model, token_prev, token_this)

# Save the model for re-use.
pickle.dump(three_model, open('models/forward_3.pkl', 'wb'))

In [6]:
# Pick a likely "first" word for a sentence.
three_word = widgets.Output(layout={'padding': '16px'})
with three_word:
    display(Markdown('## ' + next_word(three_model, 'START START START')))

three_word

Output(layout=Layout(padding='16px'))

In [7]:
def generate_three_word_context_sentence(in_model):
    last_words = ['START', 'START', 'START']
    sentence = []
    for _ in range(55):
        word = next_word(in_model, ' '.join(last_words))
        if word == 'END':
            break
        sentence.append(word)
        last_words = [last_words[-2], last_words[-1], word]

    return sentence


# Generate a random sentence.
generated_sentence = ' '.join(generate_three_word_context_sentence(three_model)) + '.'
three_sentence = widgets.Output(layout={'padding': '16px'})
with three_sentence:
    display(Markdown('## ' + generated_sentence.capitalize()))

three_sentence

Output(layout=Layout(padding='16px'))